# Movement Classification Machine Learning Sandbox

This notebook trains and evaluates multiple classification models to categorize document movements as **`ALTA`** or **`BAJA`** based on the first page OCR text.

We compare:
1. **Rule-Based Regex Classifier**
2. **TF-IDF + Naive Bayes**
3. **TF-IDF + Logistic Regression**
4. **TF-IDF + Support Vector Machine (Linear SVM)**
5. **HuggingFace Semantic Embeddings (Sentence Transformers) + Logistic Regression**

---

In [1]:
# 1. Imports
import os
import time
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sentence_transformers import SentenceTransformer

### 2. Load Dataset and Split Data
We divide the dataset according to the specified strategy:
- **Train/Test pool**: Documents in `lpb_oficios_01_06_26`, `lpb_oficios_02_06_26`, `lpb_oficios_03_06_26`, `lpb_oficios_04_06_26`, and `lpb_oficios_10_06_26`.
- **Validation Set**: Documents in `lpb_oficios_05_06_26` (held out completely for final evaluation).

In [2]:
parquet_path = "files_dataset.parquet"
if not os.path.exists(parquet_path):
    raise FileNotFoundError("files_dataset.parquet is missing. Run build_dataset.py first!")
    
df = pd.read_parquet(parquet_path)

# Print counts by directory
print("Documents count by directory:")
print(df["pdf_dir"].value_counts())

# Split Train pool vs Validation pool
val_dirs = ["lpb_oficios_05_06_26"]

train_df = df[~df["pdf_dir"].isin(val_dirs)].copy()
val_df = df[df["pdf_dir"].isin(val_dirs)].copy()

print(f"\nTrain Pool Size: {train_df.shape[0]}")
print(f"Validation Set Size: {val_df.shape[0]}")

# Perform stratified train/test split on Train pool (80% Train, 20% Test)
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(
    train_df, 
    test_size=0.2, 
    stratify=train_df["label_movement"], 
    random_state=42
)

print(f"\nStratified Split:")
print(f"  - Train Set: {train_data.shape[0]} ({train_data['label_movement'].value_counts().to_dict()})")
print(f"  - Test Set: {test_data.shape[0]} ({test_data['label_movement'].value_counts().to_dict()})")

Documents count by directory:
pdf_dir
lpb_oficios_02_06_26    35
lpb_oficios_01_06_26    33
lpb_oficios_10_06_26    31
lpb_oficios_03_06_26    10
lpb_oficios_04_06_26     4
lpb_oficios_05_06_26     2
Name: count, dtype: int64

Train Pool Size: 113
Validation Set Size: 2

Stratified Split:
  - Train Set: 90 ({'ALTA': 75, 'BAJA': 15})
  - Test Set: 23 ({'ALTA': 19, 'BAJA': 4})


### 3. Evaluate Rule-Based (Regex) Baseline
Let's see how our keyword rules perform on the test and validation datasets.

In [3]:
def rule_based_classify(text):
    txt = text.lower()
    if any(kw in txt for kw in ["elimina provisionalmente", "suspension definitiva", "dejar sin efecto"]):
        return "BAJA"
    return "ALTA"

# Evaluation function
def eval_classifier(y_true, y_pred, name, start_time):
    elapsed = time.time() - start_time
    acc = accuracy_score(y_true, y_pred) * 100
    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.2f}%")
    print(f"F1 Score (Macro): {f1:.4f}")
    print(f"Time taken: {elapsed:.4f}s")
    print(classification_report(y_true, y_pred, zero_division=0))
    return {
        "model": name,
        "accuracy": acc,
        "f1_macro": f1,
        "time_s": elapsed
    }

start = time.time()
test_preds_rule = [rule_based_classify(t) for t in test_data["extracted_text"]]
rule_metrics = eval_classifier(test_data["label_movement"], test_preds_rule, "Rule-Based Baseline (Test)", start)


=== Rule-Based Baseline (Test) ===
Accuracy: 86.96%
F1 Score (Macro): 0.6634
Time taken: 0.0017s
              precision    recall  f1-score   support

        ALTA       0.86      1.00      0.93        19
        BAJA       1.00      0.25      0.40         4

    accuracy                           0.87        23
   macro avg       0.93      0.62      0.66        23
weighted avg       0.89      0.87      0.84        23



### 4. Train Traditional ML Models (TF-IDF + Classifiers)
We vectorize the text using TF-IDF (incorporating both word unigrams and bigrams, ignoring stop words) and fit standard classifiers: Naive Bayes, Logistic Regression, and Linear SVM.

In [4]:
# Vectorizer
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_data["extracted_text"])
X_test = vectorizer.transform(test_data["extracted_text"])
y_train = train_data["label_movement"].values
y_test = test_data["label_movement"].values

ml_results = []

# A. Multinomial Naive Bayes
start = time.time()
clf_nb = MultinomialNB()
clf_nb.fit(X_train, y_train)
preds_nb = clf_nb.predict(X_test)
ml_results.append(eval_classifier(y_test, preds_nb, "TF-IDF + Naive Bayes", start))

# B. Logistic Regression
start = time.time()
clf_lr = LogisticRegression(class_weight="balanced", random_state=42)
clf_lr.fit(X_train, y_train)
preds_lr = clf_lr.predict(X_test)
ml_results.append(eval_classifier(y_test, preds_lr, "TF-IDF + Logistic Regression", start))

# C. Support Vector Machine
start = time.time()
clf_svm = SVC(kernel="linear", class_weight="balanced", random_state=42)
clf_svm.fit(X_train, y_train)
preds_svm = clf_svm.predict(X_test)
ml_results.append(eval_classifier(y_test, preds_svm, "TF-IDF + Linear SVM", start))


=== TF-IDF + Naive Bayes ===
Accuracy: 82.61%
F1 Score (Macro): 0.4524
Time taken: 0.0047s
              precision    recall  f1-score   support

        ALTA       0.83      1.00      0.90        19
        BAJA       0.00      0.00      0.00         4

    accuracy                           0.83        23
   macro avg       0.41      0.50      0.45        23
weighted avg       0.68      0.83      0.75        23


=== TF-IDF + Logistic Regression ===
Accuracy: 100.00%
F1 Score (Macro): 1.0000
Time taken: 0.0171s
              precision    recall  f1-score   support

        ALTA       1.00      1.00      1.00        19
        BAJA       1.00      1.00      1.00         4

    accuracy                           1.00        23
   macro avg       1.00      1.00      1.00        23
weighted avg       1.00      1.00      1.00        23




=== TF-IDF + Linear SVM ===
Accuracy: 100.00%
F1 Score (Macro): 1.0000
Time taken: 0.0733s
              precision    recall  f1-score   support

        ALTA       1.00      1.00      1.00        19
        BAJA       1.00      1.00      1.00         4

    accuracy                           1.00        23
   macro avg       1.00      1.00      1.00        23
weighted avg       1.00      1.00      1.00        23



### 5. Train Advanced Semantic Model (Sentence Transformers + SVM)
Here we load a local multilingual Transformer (`paraphrase-multilingual-MiniLM-L12-v2`) to extract dense sentence embeddings from the document texts. This captures semantic similarity in Spanish rather than just exact word counts. Then we train a classifier on top of the embeddings.

In [5]:
print("Loading multilingual Sentence Transformer model...")
model_name = "local_models/paraphrase-multilingual-MiniLM-L12-v2"
if not os.path.exists(model_name):
    model_name = "paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(model_name)

print("Generating sentence embeddings for train and test sets...")
start_embed = time.time()
X_train_emb = embedder.encode(train_data["extracted_text"].tolist(), show_progress_bar=True)
X_test_emb = embedder.encode(test_data["extracted_text"].tolist(), show_progress_bar=True)
print(f"Embeddings generated in {time.time() - start_embed:.2f}s")

# Fit Classifier on top of embeddings
start = time.time()
clf_emb = SVC(kernel="linear", class_weight="balanced", random_state=42)
clf_emb.fit(X_train_emb, y_train)
preds_emb = clf_emb.predict(X_test_emb)
ml_results.append(eval_classifier(y_test, preds_emb, "Semantic Embeddings + SVM", start))

Loading multilingual Sentence Transformer model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating sentence embeddings for train and test sets...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated in 6.30s

=== Semantic Embeddings + SVM ===
Accuracy: 69.57%
F1 Score (Macro): 0.5165
Time taken: 0.0090s
              precision    recall  f1-score   support

        ALTA       0.83      0.79      0.81        19
        BAJA       0.20      0.25      0.22         4

    accuracy                           0.70        23
   macro avg       0.52      0.52      0.52        23
weighted avg       0.72      0.70      0.71        23



### 6. Validation Set Comparison (lpb_oficios_05_06_26)
Let's evaluate all models on the completely independent validation set (`lpb_oficios_05_06_26`).

In [6]:
X_val_text = val_df["extracted_text"].tolist()
y_val = val_df["label_movement"].values

print(f"Evaluating validation set of size {len(y_val)}...")

# 1. Rule-based
preds_val_rule = [rule_based_classify(t) for t in X_val_text]
acc_val_rule = accuracy_score(y_val, preds_val_rule) * 100

# TF-IDF transformation
X_val_tfidf = vectorizer.transform(X_val_text)

# 2. Naive Bayes
preds_val_nb = clf_nb.predict(X_val_tfidf)
acc_val_nb = accuracy_score(y_val, preds_val_nb) * 100

# 3. Logistic Regression
preds_val_lr = clf_lr.predict(X_val_tfidf)
acc_val_lr = accuracy_score(y_val, preds_val_lr) * 100

# 4. SVM
preds_val_svm = clf_svm.predict(X_val_tfidf)
acc_val_svm = accuracy_score(y_val, preds_val_svm) * 100

# 5. Semantic Embeddings SVM
X_val_emb = embedder.encode(X_val_text, show_progress_bar=False)
preds_val_emb = clf_emb.predict(X_val_emb)
acc_val_emb = accuracy_score(y_val, preds_val_emb) * 100

# Compilation
val_comparison = {
    "Model": [
        "Rule-Based Regex", 
        "TF-IDF + Naive Bayes", 
        "TF-IDF + Logistic Regression", 
        "TF-IDF + Linear SVM", 
        "Semantic Embeddings + SVM"
    ],
    "Validation Accuracy (%)": [
        acc_val_rule, 
        acc_val_nb, 
        acc_val_lr, 
        acc_val_svm, 
        acc_val_emb
    ]
}
df_val_comp = pd.DataFrame(val_comparison)
display(df_val_comp)

Evaluating validation set of size 2...


,Model,Validation Accuracy (%)
0,Rule-Based Regex,0.0
1,TF-IDF + Naive Bayes,0.0
2,TF-IDF + Logistic Regression,100.0
3,TF-IDF + Linear SVM,100.0
4,Semantic Embeddings + SVM,100.0
